# Phase 16 — DQN Terrain Generalization

This is the single user-facing entry point for Phases 16.1–16.4. Set the switches in the next cell and choose **Run All**. Only enabled phases run. The calculation modules remain in `3D_RL_DQN/*.py` so they can be tested and reused without duplicating scientific logic in the notebook.

Official Phase 16.3/16.4 training requires CUDA. Phase 16.4 writes a resumable checkpoint every evaluation interval; after a kernel or machine restart, leave `RESUME_PHASE_16_4 = True` and run again. Closing only the browser tab does not stop a live Jupyter server/kernel, but kernel shutdown, sleep, or reboot stops active computation.

In [ ]:
# ========================= USER SWITCHES =========================
RUN_PHASE_16_1 = False
RUN_PHASE_16_2 = False
RUN_PHASE_16_3 = False
RUN_PHASE_16_4 = False

# Official training protection and Phase 16.4 restart behavior.
REQUIRE_CUDA_FOR_TRAINING = True
RESUME_PHASE_16_4 = True

# Set a smoke switch True only for a short, nonofficial pipeline check.
PHASE_16_3_SMOKE = False
PHASE_16_4_SMOKE = False

# Condition-specific DQN. Official Phase 16.4 defaults are 25 m and 5 degrees.
SPATIAL_RESOLUTION_M = 25.0
HEADING_SPACING_DEG = 5.0
# ================================================================


In [ ]:
# Repository and kernel preflight. This cell performs no phase computation.
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate.parent != candidate and not (candidate / '3D_RL_DQN').is_dir():
    candidate = candidate.parent
if not (candidate / '3D_RL_DQN').is_dir() or not (candidate / '3D_0827').is_dir():
    raise RuntimeError('Could not locate the glider_hybrid_control repository root')
WORKSPACE_ROOT = candidate
for source_dir in (WORKSPACE_ROOT / '3D_RL_DQN', WORKSPACE_ROOT / '3D_0827'):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))

import torch
training_requested = (
    (RUN_PHASE_16_3 and not PHASE_16_3_SMOKE)
    or (RUN_PHASE_16_4 and not PHASE_16_4_SMOKE)
)
print('Workspace:', WORKSPACE_ROOT)
print('Kernel Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if training_requested and REQUIRE_CUDA_FOR_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('Official Phase 16.3/16.4 training requires a CUDA-enabled kernel')

PHASE_RESULTS = {}


## Phase 16.1 — Common attacker best-response interface

In [ ]:
if RUN_PHASE_16_1:
    from phase1_validation import run_phase1
    PHASE_RESULTS['16.1'] = run_phase1()
else:
    print('Phase 16.1 skipped')


## Phase 16.2 — Terrain-aware observation and visualization

In [ ]:
if RUN_PHASE_16_2:
    from phase2_validation import run_phase2
    PHASE_RESULTS['16.2'] = str(run_phase2())
else:
    print('Phase 16.2 skipped')


## Phase 16.3 — Single-terrain DQN sanity experiment

In [ ]:
if RUN_PHASE_16_3:
    from phase3_single_terrain import run_phase3
    phase3_device = 'cpu' if PHASE_16_3_SMOKE else ('cuda' if REQUIRE_CUDA_FOR_TRAINING else 'auto')
    PHASE_RESULTS['16.3'] = str(run_phase3(
        smoke=PHASE_16_3_SMOKE,
        device_override=phase3_device,
        spatial_resolution_m=SPATIAL_RESOLUTION_M,
        heading_spacing_deg=HEADING_SPACING_DEG,
    ))
else:
    print('Phase 16.3 skipped')


## Phase 16.4 — Multi-terrain generalized DQN

The official configuration trains one shared DQN for 60,000 episodes per seed on 27 training scenarios and evaluates 18 held-out simple-terrain scenarios. Progress and resumable state are saved every 1,500 episodes.

In [ ]:
if RUN_PHASE_16_4:
    from phase4_multi_terrain import run_phase4
    phase4_device = 'cpu' if PHASE_16_4_SMOKE else ('cuda' if REQUIRE_CUDA_FOR_TRAINING else 'auto')
    PHASE_RESULTS['16.4'] = str(run_phase4(
        smoke=PHASE_16_4_SMOKE,
        resume=RESUME_PHASE_16_4,
        device=phase4_device,
        require_cuda=REQUIRE_CUDA_FOR_TRAINING and not PHASE_16_4_SMOKE,
        spatial_resolution_m=SPATIAL_RESOLUTION_M,
        heading_spacing_deg=HEADING_SPACING_DEG,
    ))
else:
    print('Phase 16.4 skipped')


## Run summary

In [ ]:
if PHASE_RESULTS:
    print('Completed enabled phases:')
    for phase, result in PHASE_RESULTS.items():
        print(f'  {phase}: {result}')
else:
    print('No phase was enabled. Set one or more RUN_PHASE_16_* switches to True and Run All.')
